# AirBnB NYC Analytics Project

* Project by Nhi Bui · Villanova University · [GitHub](https://github.com/nhibui23/airbnb-nyc-product-analytics-project) · [LinkedIn](https://linkedin.com/in/nhiuyenbui)

## 06. Synthesis & Recommendation to Leadership

> "Should Airbnb build HostLens?"

This notebook combines findings from Notebooks 03 (guest lens), 04 (host lens), and 05 (routing threshold) into one single recommendation. It answers the 3 questions Airbnb leadership needs to make the build decision:

1. Is the target market real and large enough?
2. Does HostLens have a defensible basis for what it recommends?
3. Does the prototype demonstrate that the concept works?

## 1. What we found - The build decision

**On market size (host lens):**
- 6,412 listings (10.1% of the dataset) fall into the target segment: 5-star rating, less than 50% occupancy, $620/night average
- Aggregate revenue gap in this segment is $735M if lifted to 70% occupancy
- Availability tier significantly affects rating (ANOVA p < 0.0001). Selective hosts (0-90 days available) and near-year-round hosts both outperform the middle group

**On what drives guest behavior (guest lens):**
- Instant Book has no measurable effect on ratings (Welch's t-test, p > 0.05)
- Host verification has no measurable effect on ratings (p = 0.9114)
- Price tier does affect ratings. Very High priced listings underperform Medium and High tiers (ANOVA p = 0.0008, confirmed by Tukey's HSD)

**On the routing threshold (review count):**
- Occupancy rises from 50.5% to 55.2% between the 0-10 and 11-50 review bins (significant)
- Occupancy is flat or slightly declining past 50 reviews
- The flattening point is around 50 reviews, much lower than the commonly cited threshold of several hundred

### 2. Why these patterns may exist

**Instant Book and host verification show no effect** likely because both are convenience and trust features, not quality features. Guests use them to filter but don't rate their stays differently because of them.

**Very High priced listings underperform on rating** likely because premium guests have higher expectations. A $1,200/night listing is judged against luxury hotels, not against other Airbnbs.

**The target segment is concentrated in premium listings** because low-priced, high-availability listings tend to book steadily. Premium listings depend on specific demand windows (events, business travel, wedding season) and sit empty when those windows are missed.

**Reviews flatten early because trust builds fast.** Guests scanning a listing likely stop caring about additional reviews once a threshold of social proof is reached. Fifty reviews at 5 stars appears to be enough for that trust signal.

## 3. The build decision

**Recommendation: build HostLens.**

All three validation questions were answered affirmatively:

**Market size (Q1):** The target segment holds 6,412 listings and $735M of unused revenue in NYC alone. This is a large enough opportunity to justify product investment.

**Defensible recommendation basis (Q2):** Price and review count are proven booking drivers. Instant Book and verification are not. HostLens has evidence-backed rules for what to recommend and what to avoid recommending.

**Prototype viability (Q3):** The live prototype demonstrates that the routing logic works end to end. A host from the target segment can load in, receive personalized AI recommendations, and see event-based positioning suggestions, all within a working product interface.

2 assumptions built into current Airbnb host guidance didn't survive the analysis. Instant Book and host verification do not measurably affect ratings. Airbnb should not promote these features to hosts as booking-improvement tools.

## 4. Proposed next steps

**A/B tests to run before scaling HostLens:**

| Test | Hypothesis | Metric | Decision rule |
|---|---|---|---|
| HostLens onboarding | Hosts in the target segment who receive HostLens recommendations will book more nights than a control group | Booked nights per month over 90 days | Ship if lift ≥ 10% at p < 0.05 |
| Review threshold guidance | Hosts under 50 reviews who see "focus on getting more reviews" prompts will earn more reviews than a control group | New reviews per month over 60 days | Ship if lift ≥ 15% at p < 0.05 |
| Instant Book demotion | Removing Instant Book as a promoted feature will not reduce booking rate | Booking rate for affected listings | Keep the change if no negative effect at p < 0.05 |

**Data needed for stronger conclusions:**
- Actual booking data (this analysis relies on an availability-based occupancy proxy)
- Real Airbnb ratings, which cluster between 4.5 and 5.0 rather than the flat distribution seen here
- Longitudinal data to distinguish causation from correlation in the review count finding
- Text and photo data to test positioning changes against HostLens recommendations

## 5. Limitations

* The dataset is observational, not experimental

→ Findings show correlation, not causation.

* Occupancy is a proxy calculated as `(365 - availability 365) / 365`

→ It reflects host settings, not actual bookings

* Ratings in this dataset are nearly uniformly distributed from 2 to 5 stars, which does not match the shape of real Airbnb data.

→ Effect sizes should be read as directional rather than exact. 

* Only 9 listings in this dataset have more than 500 reviews, which limits testing of the review count hypothesis

* The unused revenue segment's average rating of 5.00 is a dataset artifact. A live filter would surface a broader 4.5-5.0 range.

## 6. Handoff

The analysis findings above transfer into three downstream artifacts:

**SQL layer (`/sql`):** All notebook logic (proxy calculations, segment filters, borough rollups) is translated into commented PostgreSQL queries in `02_product_queries.sql`. These queries can be run directly against the underlying tables to reproduce the notebook findings without Python.

**Power BI dashboard (`/powerbi`):** A 2-page executive-facing report. Page 1 (Guest View) presents the Notebook 03 findings on which guest-facing features actually matter. Page 2 (Host View) presents the market size case from Notebook 04, including the $735M revenue gap and the geographic concentration of the target segment.

**HostLens prototype (`/prototype`):** A Streamlit app powered by the Claude API. Input is a host from the target segment. Output is a set of prioritized recommendations, routed based on the 50-review threshold from Notebook 05. This is the working proof of concept that leadership can interact with to validate the product feel before committing to build.